# Notebook 5: specification robustness

Stochastic-volatility, distributional and Minnesota-prior sensitivity under the selected conditional mean.

In [1]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *

import numpy as np
import pymc as pm
import pytensor.tensor as pt
from pytensor import scan as pt_scan
import arviz as az

QUICK = quick_mode()
selection = load_result("nb0_selection")["selection"]
P = int(selection["p"])
K = int(selection["k"])

d = load_data(network="geographic")
Y_train = d["Y_train"].to_numpy()
N = d["N"]
W = knn_sparsify(d["networks"]["geographic"], K)
STAGES = [2] * P

X, y, names = build_design(Y_train, W, p=P, stages=STAGES, mode="global_gnar", h=1)
tidx = build_time_index(Y_train, p=P, h=1)
assert len(names) == 3 * P

kw = dict(NUTS_KW)
if QUICK:
    kw.update(draws=2, tune=2, chains=1, cores=1)

## Log-variance process sensitivity

In [2]:
def build_rw_logvar(X, y, tidx, names, N):
    """Selected GNAR mean with a random-walk log variance."""
    n_time = len(tidx)
    row_time = np.repeat(np.arange(n_time), N)
    mu0, sd0 = minnesota_prior(names)
    with pm.Model() as model:
        beta = pm.Normal("beta", mu=mu0, sigma=sd0, shape=len(names))
        sigma_h = pm.HalfNormal("sigma_h", 0.5)
        h0 = pm.Normal("h0", -2.0, 1.0)
        z = pm.Normal("h_innov", 0.0, 1.0, shape=n_time - 1)
        h = pm.Deterministic("h", pt.concatenate([h0[None], h0 + pt.cumsum(sigma_h * z)]))
        pm.Normal("y_obs", mu=pt.dot(X, beta), sigma=pt.exp(h / 2)[row_time], observed=y)
    return model

def build_fattail_vol(X, y, tidx, names, N):
    """Selected GNAR mean with Student-t log-variance innovations."""
    n_time = len(tidx)
    row_time = np.repeat(np.arange(n_time), N)
    mu0, sd0 = minnesota_prior(names)
    with pm.Model() as model:
        beta = pm.Normal("beta", mu=mu0, sigma=sd0, shape=len(names))
        m_h = pm.Normal("m_h", -2.0, 1.0)
        phi = pm.Uniform("phi", -0.99, 0.99)
        sigma_h = pm.HalfNormal("sigma_h", 0.5)
        nu_vol = pm.Gamma("nu_vol", alpha=2.0, beta=0.1)
        z = pm.StudentT("h_innov", nu=nu_vol, mu=0.0, sigma=1.0, shape=n_time)

        def step(z_t, h_prev, m, ph, s):
            return m + ph * (h_prev - m) + s * z_t

        h0 = m_h + sigma_h * z[0] / pt.sqrt(1 - phi**2)
        h_seq, _ = pt_scan(
            fn=step, sequences=[z[1:]], outputs_info=[h0],
            non_sequences=[m_h, phi, sigma_h],
        )
        h = pm.Deterministic("h", pt.concatenate([h0[None], h_seq]))
        pm.Normal("y_obs", mu=pt.dot(X, beta), sigma=pt.exp(h / 2)[row_time], observed=y)
    return model

with build_gnar_sv(X, y, tidx, names, N) as model:
    idata_ar1 = pm.sample(**kw)
with build_rw_logvar(X, y, tidx, names, N) as model:
    idata_rw = pm.sample(**kw)
with build_fattail_vol(X, y, tidx, names, N) as model:
    idata_ft = pm.sample(**kw)

diag_ar1 = mcmc_diagnostics(idata_ar1)
diag_rw = mcmc_diagnostics(idata_rw)
diag_ft = mcmc_diagnostics(idata_ft)

phi = idata_ar1.posterior["phi"].values.ravel()
h_rw = idata_rw.posterior["h"].mean(("chain", "draw")).values
nu_vol = idata_ft.posterior["nu_vol"].values.ravel()

print(f"AR(1) log-variance phi mean={phi.mean():.5f}")
print(f"random-walk log-variance drift={h_rw[-1]-h_rw[0]:+.5f}")
print(f"Student-t volatility innovation df mean={nu_vol.mean():.5f}")

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

## Minnesota prior centring

In [3]:
def build_const_centred(rw_centre):
    """Constant-variance selected GNAR with a specified first-lag prior centre."""
    minn = dict(MINN)
    minn["rw_centre"] = rw_centre
    mu0, sd0 = minnesota_prior(names, minn=minn)
    with pm.Model() as model:
        beta = pm.Normal("beta", mu=mu0, sigma=sd0, shape=len(names))
        sigma = pm.HalfNormal("sigma", 1.0)
        pm.Normal("y_obs", mu=pt.dot(X, beta), sigma=sigma, observed=y)
    return model

def _loo_field(obj, *keys):
    """Extract an ArviZ ELPDData field across supported return types."""
    for key in keys:
        try:
            return obj[key]
        except (KeyError, TypeError, IndexError, AttributeError):
            pass
        if hasattr(obj, key):
            return getattr(obj, key)
    raise KeyError(keys)

with build_const_centred(1.0) as model:
    idata_rwcentre = pm.sample(**kw)
with build_const_centred(0.0) as model:
    idata_zerocentre = pm.sample(**kw)

if QUICK:
    elpd_rw = elpd_zero = pareto_rw = pareto_zero = np.nan
else:
    loo_rw = az.loo(idata_rwcentre, pointwise=True)
    loo_zero = az.loo(idata_zerocentre, pointwise=True)
    elpd_rw = float(_loo_field(loo_rw, "elpd", "elpd_loo"))
    elpd_zero = float(_loo_field(loo_zero, "elpd", "elpd_loo"))
    pareto_rw = float(np.nanmax(np.asarray(_loo_field(loo_rw, "pareto_k"), dtype=float)))
    pareto_zero = float(np.nanmax(np.asarray(_loo_field(loo_zero, "pareto_k"), dtype=float)))

beta_rw = idata_rwcentre.posterior["beta"].mean(("chain", "draw")).values
beta_zero = idata_zerocentre.posterior["beta"].mean(("chain", "draw")).values
print(f"ELPD difference, centre 1 minus centre 0: {elpd_rw-elpd_zero:+.5f}")
print(f"maximum posterior-mean coefficient shift: {np.max(np.abs(beta_rw-beta_zero)):.5f}")

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [beta, sigma]


Output()

In [4]:
save_result("nb5_specification_robustness", {
    "selection": selection,
    "log_variance": {
        "ar1_phi_mean": float(phi.mean()),
        "ar1_phi_lo95": float(np.percentile(phi, 2.5)),
        "ar1_phi_hi95": float(np.percentile(phi, 97.5)),
        "rw_drift": float(h_rw[-1] - h_rw[0]),
        "student_t_df_mean": float(nu_vol.mean()),
        "diagnostics": {"ar1": diag_ar1, "random_walk": diag_rw, "student_t": diag_ft},
    },
    "minnesota_centre": {
        "elpd_centre1": float(elpd_rw),
        "elpd_centre0": float(elpd_zero),
        "elpd_difference": float(elpd_rw - elpd_zero),
        "max_pareto_k_centre1": float(pareto_rw),
        "max_pareto_k_centre0": float(pareto_zero),
        "max_posterior_mean_beta_difference": float(np.max(np.abs(beta_rw-beta_zero))),
        "diagnostics_centre1": mcmc_diagnostics(idata_rwcentre),
        "diagnostics_centre0": mcmc_diagnostics(idata_zerocentre),
    },
    "config": run_config(p=P, stages=STAGES, network="geographic", k=K,
                         max_stage=2, purpose="specification_robustness"),
    "quick": QUICK,
})
print("saved nb5_specification_robustness")

saved nb5_specification_robustness
